# Checkpoint Data Analysis

This notebook reproduces the comprehensive analysis of checkpoint data performed during the debugging session.


In [ ]:
import polars as pl
from pathlib import Path
from collections import defaultdict
import json

# Set cache directory
CACHE_DIR = "data/.cache/wikiqa"

## 1. Discover Available Data

In [ ]:
cache_path = Path(CACHE_DIR)
parquet_files = list(cache_path.glob('*.parquet'))

print(f"Found {len(parquet_files)} parquet files in {cache_path}")
print("\nFirst 10 files:")
for file in sorted(parquet_files)[:10]:
    print(f"  {file.name}")

if len(parquet_files) > 10:
    print(f"  ... and {len(parquet_files) - 10} more")

## 2. Basic Record Count Analysis

In [ ]:
total_records = 0
file_data = []

print("File record counts:")
for file in sorted(parquet_files):
    try:
        df = pl.read_parquet(file)
        records = len(df)
        total_records += records
        file_data.append((file.name, records))
        print(f"{file.name}: {records:,} records")
    except Exception as e:
        print(f"❌ Error reading {file.name}: {e}")

print(f"\nTotal datapoints: {total_records:,}")
print(f"Average records per file: {total_records / len(file_data):.1f}")

## 3. Data Structure Inspection

In [ ]:
# Sample a few files to understand data structure
sample_files = [
    '20_enhanced_rag_False_200_1758048508_9fb41597.parquet',
    '5_enhanced_rag_False_2_1758139804_5ed886a1.parquet', 
    '1_full_context_False_200_1758060460_6ebf2140.parquet'
]

for filename in sample_files:
    file_path = cache_path / filename
    if file_path.exists():
        print(f"\n=== {filename} ===")
        df = pl.read_parquet(file_path)
        print(f"Shape: {df.shape}")
        print(f"Columns ({len(df.columns)}): {list(df.columns)}")
        
        # Check for retrieval-related columns
        retrieval_cols = [col for col in df.columns if 'retrieval' in col.lower()]
        print(f"Retrieval columns: {retrieval_cols}")
        
        # Show first record structure
        if len(df) > 0:
            first_record = df.row(0, named=True)
            print(f"Sample keys: {list(first_record.keys())[:10]}...")
    else:
        print(f"File not found: {filename}")

## 4. Retrieval Data Completeness Analysis

In [ ]:
files_with_retrieval_metrics = []
files_missing_retrieval_metrics = []
full_context_files = []

print("Retrieval data analysis:")

for file in sorted(parquet_files):
    try:
        df = pl.read_parquet(file)
        total_records = len(df)
        
        # Check if this is full_context experiment
        is_full_context = 'full_context' in file.name
        
        if is_full_context:
            full_context_files.append((file.name, total_records))
            print(f"🔵 FULL_CONTEXT: {file.name} - {total_records} records")
            continue
            
        # Check for retrieval metric columns (flattened format)
        retrieval_metric_cols = [col for col in df.columns if col.startswith('retrieval_benchmarks_')]
        
        if retrieval_metric_cols:
            # Check if metrics have actual values (not all null)
            any_non_null = False
            for col in retrieval_metric_cols[:3]:  # Check first few columns
                if df.filter(pl.col(col).is_not_null()).height > 0:
                    any_non_null = True
                    break
            
            if any_non_null:
                files_with_retrieval_metrics.append((file.name, total_records, len(retrieval_metric_cols)))
                print(f"✅ RAG WITH METRICS: {file.name} - {total_records} records, {len(retrieval_metric_cols)} metrics")
            else:
                files_missing_retrieval_metrics.append((file.name, total_records, 'All metrics null'))
                print(f"❌ RAG NULL METRICS: {file.name} - {total_records} records (all metrics null)")
        else:
            files_missing_retrieval_metrics.append((file.name, total_records, 'No metric columns'))
            print(f"❌ RAG NO METRICS: {file.name} - {total_records} records (no metric columns)")
            
    except Exception as e:
        print(f"❌ Error reading {file.name}: {e}")

## 5. Summary Statistics

In [ ]:
total_with_metrics = sum(count for _, count, _ in files_with_retrieval_metrics)
total_missing_metrics = sum(count for _, count, _ in files_missing_retrieval_metrics)
total_full_context = sum(count for _, count in full_context_files)

print("=== COMPLETENESS SUMMARY ===")
print(f"Total files: {len(parquet_files)}")
print(f"Total datapoints: {total_records:,}")
print()
print(f"✅ RAG files with retrieval metrics: {len(files_with_retrieval_metrics)}")
print(f"❌ RAG files missing retrieval metrics: {len(files_missing_retrieval_metrics)}")
print(f"🔵 Full context files (no metrics needed): {len(full_context_files)}")
print()
print(f"Datapoints with retrieval metrics: {total_with_metrics:,}")
print(f"Datapoints missing retrieval metrics: {total_missing_metrics:,}")
print(f"Full context datapoints: {total_full_context:,}")
print()
print(f"Percentage with complete data: {(total_with_metrics + total_full_context) / total_records * 100:.1f}%")
if total_missing_metrics > 0:
    print(f"Percentage missing retrieval data: {total_missing_metrics / total_records * 100:.1f}%")

## 6. Experiment Configuration Breakdown

In [ ]:
# Group by experiment configuration
config_data = defaultdict(lambda: {'files': [], 'total_records': 0})

for file in sorted(parquet_files):
    try:
        df = pl.read_parquet(file)
        total_records = len(df)
        
        # Parse experiment configuration from filename
        filename = file.name.replace('.parquet', '')
        parts = filename.split('_')
        
        if len(parts) >= 4:
            k = parts[0]
            retrieval_kind = parts[1] + '_' + parts[2]  # basic_rag, enhanced_rag, full_context
            rerank = parts[3]
            n_questions = parts[4] if len(parts) > 4 else 'unknown'
            
            config_key = f'k={k}, {retrieval_kind}, rerank={rerank}, n={n_questions}'
            config_data[config_key]['files'].append(file.name)
            config_data[config_key]['total_records'] += total_records
            
    except Exception as e:
        print(f'Error reading {file.name}: {e}')

# Display results
print("=== EXPERIMENT CONFIGURATION BREAKDOWN ===")
for config, data in sorted(config_data.items()):
    print(f"{config}:")
    print(f"  Files: {len(data['files'])}")
    print(f"  Total datapoints: {data['total_records']:,}")
    
    if len(data['files']) <= 3:
        for filename in sorted(data['files']):
            print(f"    - {filename}")
    else:
        for filename in sorted(data['files'])[:2]:
            print(f"    - {filename}")
        print(f"    ... and {len(data['files']) - 2} more")
    print()

## 7. Summary by Approach Type

In [ ]:
# Summary by approach type
approach_summary = defaultdict(lambda: {'files': 0, 'records': 0})
for config, data in config_data.items():
    if 'basic_rag' in config:
        approach_summary['Basic RAG']['files'] += len(data['files'])
        approach_summary['Basic RAG']['records'] += data['total_records']
    elif 'enhanced_rag' in config:
        approach_summary['Enhanced RAG']['files'] += len(data['files'])
        approach_summary['Enhanced RAG']['records'] += data['total_records']
    elif 'full_context' in config:
        approach_summary['Full Context']['files'] += len(data['files'])
        approach_summary['Full Context']['records'] += data['total_records']

print("=== SUMMARY BY APPROACH TYPE ===")
for approach, data in approach_summary.items():
    print(f"{approach}: {data['files']} files, {data['records']:,} datapoints")

## 8. Load Consolidated Dataset (Optional)

In [ ]:
# Load all checkpoint data into a single DataFrame
print(f"Loading {len(parquet_files)} parquet files into consolidated DataFrame...")

all_dfs = []
for file in parquet_files:
    try:
        df = pl.read_parquet(file)
        # Add source file for tracking
        df = df.with_columns(pl.lit(file.name).alias("source_file"))
        all_dfs.append(df)
    except Exception as e:
        print(f"❌ Error loading {file.name}: {e}")

if all_dfs:
    # Concatenate all DataFrames
    print("Concatenating data...")
    consolidated_df = pl.concat(all_dfs, how="diagonal")  # diagonal handles different schemas
    
    print(f"✅ Loaded {len(consolidated_df):,} total records from {len(all_dfs)} files")
    print(f"Final shape: {consolidated_df.shape}")
    print(f"Columns: {consolidated_df.columns}")
else:
    print("❌ No data files could be loaded")
    consolidated_df = None

## 9. Sample Data Exploration

In [ ]:
if consolidated_df is not None:
    print("=== SAMPLE DATA EXPLORATION ===")
    
    # Show basic statistics
    print("Unique values per key column:")
    for col in ['k', 'retrieval_kind', 'rerank']:
        if col in consolidated_df.columns:
            unique_vals = consolidated_df[col].unique().to_list()
            print(f"  {col}: {unique_vals}")
    
    # Show sample record
    print("\nFirst record sample:")
    if len(consolidated_df) > 0:
        sample_record = consolidated_df.row(0, named=True)
        for key, value in list(sample_record.items())[:10]:
            print(f"  {key}: {str(value)[:100]}{'...' if len(str(value)) > 100 else ''}")
    
    # Available for further analysis
    print(f"\n✅ consolidated_df is ready for analysis with {len(consolidated_df):,} records")
else:
    print("❌ Consolidated DataFrame not available")